# ResNet50 ImageNet — Colab Training
## Jenks Pruning + Post-Prune EMA + Optional HPO

**GPU note**: High-RAM Colab does **not** make training faster — GPU tier is the same.
Use **Colab Pro+ with A100** for speed (~8-10x faster than T4).

### Files to upload to Google Drive (into one folder, e.g. `MyDrive/JenksTests/`):
- `custom_optimizer.py`
- `custom_schedulers.py`
- `training_loop_v2.py`
- `cuda_helpers.py` ← contains embedded CUDA source; compiled automatically at runtime

Then set `REPO_PATH` in Cell 2 to that folder and run top-to-bottom.


In [ ]:
# Cell 1 — Install dependencies
!pip install -q datasets optuna torchvision torchmetrics huggingface_hub jenkspy pynvml


In [ ]:
# Cell 2 — Mount Google Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

import sys, os

REPO_PATH  = '/content/drive/MyDrive/JenksTests'   # <-- ADJUST THIS
OUTPUT_DIR = '/content/drive/MyDrive/ResNet50_output'
MODEL_DIR  = '/content/drive/MyDrive/models'

sys.path.insert(0, REPO_PATH)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,  exist_ok=True)
os.makedirs('models',   exist_ok=True)
print('Drive mounted.  REPO_PATH:', REPO_PATH)


In [ ]:
# Cell 3 — Verify Drive files and patch cuda_helpers for this GPU's SM arch
import pathlib, re, torch

# ── Check every required file is present ─────────────────────────────────────
required = ['custom_optimizer.py', 'custom_schedulers.py',
            'training_loop_v2.py', 'cuda_helpers.py']
missing = [f for f in required if not (pathlib.Path(REPO_PATH) / f).exists()]
if missing:
    raise FileNotFoundError(f'Missing from {REPO_PATH}: {missing}')
print('All required files found:')
!ls -lh '{REPO_PATH}'

# ── Patch cuda_helpers.py to use the correct GPU SM arch ─────────────────────
# The original file has a hardcoded sm_86 (RTX 3090 / A10).  This auto-detects
# the arch for whatever GPU Colab assigns (A100=sm_80, T4=sm_75, etc.).
cap    = torch.cuda.get_device_capability()
sm_str = f'sm_{cap[0]}{cap[1]}'
ar_str = f'compute_{cap[0]}{cap[1]}'

src = (pathlib.Path(REPO_PATH) / 'cuda_helpers.py').read_text()
src = re.sub(
    r'"-gencode",\s*"arch=compute_\d+,code=sm_\d+"',
    f'"-gencode", "arch={ar_str},code={sm_str}"',
    src,
)
patched_path = pathlib.Path('/content/cuda_helpers.py')
patched_path.write_text(src)
sys.path.insert(0, '/content')   # /content shadows REPO_PATH for cuda_helpers
print(f'cuda_helpers.py patched for {torch.cuda.get_device_name(0)}  ({sm_str})')


In [ ]:
# Cell 4 — HuggingFace authentication (ImageNet requires licence acceptance)
import getpass
from huggingface_hub import login
hf_token = getpass.getpass('HuggingFace token: ')
login(token=hf_token, add_to_git_credential=False)
print('Logged in.')


In [ ]:
# Cell 5 — Imports  (cuda_helpers compiles CUDA kernels here; takes ~1 min first run)
import random
import torch
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32        = True
torch.backends.cudnn.benchmark         = True

import torch.nn as nn
from datetime import datetime
from torch.utils.data import DataLoader, IterableDataset
from torch.utils.tensorboard import SummaryWriter
from torchmetrics import Accuracy
from torchmetrics.classification import MulticlassAccuracy
from torchvision import transforms
from torchvision.models import resnet50
from datasets import load_dataset

print('Importing cuda_helpers (compiling CUDA kernels)...')
import cuda_helpers   # triggers load_inline compilation
print('cuda_helpers OK')

from custom_optimizer import Prune_Score_Select, init_network
import custom_optimizer
print('custom_optimizer OK')

from custom_schedulers import init_lr_weight_decay, WarmupAutoJenks
print('custom_schedulers OK')

from training_loop_v2 import train_val_loop_HPO
print('training_loop_v2 OK')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nDevice : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
# Cell 6 — Configuration
# ── HPO ───────────────────────────────────────────────────────────────────────
N_HPO_TRIALS          = 0        # set > 0 to run Optuna HPO before full run
HPO_EPOCHS            = 15       # proxy epochs per trial
HPO_SAMPLES_PER_EPOCH = 50_000   # images per HPO epoch (cap for speed)

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS        = 400
prune_epoch   = 350
BATCH_SIZE    = 128    # A100: 256 comfortable; T4: 64
NUM_WORKERS   = 4
accum_steps   = 4      # effective batch = BATCH_SIZE * accum_steps
BUFFER_SIZE   = 2000   # HF shuffle buffer; increase on high-RAM Colab

# ── Default hyperparameters (replaced by HPO if N_HPO_TRIALS > 0) ─────────────
learning_rate     = 5e-2
weight_decay      = 5e-4
bias_weight_decay = 2e-4
momentum          = 0.99
label_smoothing   = 0.1

# ── Fixed settings ────────────────────────────────────────────────────────────
warmup_epochs     = 10
nestrov           = True
bias_lr           = True
prune_ratio       = 0.5
one_shot          = True
mask              = True
bias_prune        = False
kill_velocity     = False
one_update        = True
prune_between     = 5
min_epochs        = 300
gsm_lr_boundaries = [200, 230, 260]

# ── EMA: no tracking before pruning, full tracking after ─────────────────────
USE_EMA         = True
EMA_DECAY       = 0.9999
EMA_START_EPOCH = prune_epoch

custom_optimizer.MIXUP           = True
custom_optimizer.MIXUP_OFF_EPOCH = EPOCHS - 20

print(f'Effective batch : {BATCH_SIZE * accum_steps}')
print(f'EMA active      : epochs {EMA_START_EPOCH}–{EPOCHS} (after pruning only)')


In [ ]:
# Cell 7 — Dataset and transforms
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                  std=[0.229, 0.224, 0.225])
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.ToTensor(),
    normalize,
])
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalize,
])

class HFStreamingImageNet(IterableDataset):
    def __init__(self, split, transform=None, shuffle=False, buffer_size=2000):
        self.split, self.transform = split, transform
        self.shuffle, self.buffer_size = shuffle, buffer_size

    def __iter__(self):
        ds = load_dataset('ILSVRC/imagenet-1k', split=self.split, streaming=True)
        wi = torch.utils.data.get_worker_info()
        if wi is not None:
            ds = ds.shard(num_shards=wi.num_workers, index=wi.id)
        if self.shuffle:
            ds = ds.shuffle(seed=random.randint(0, 99999), buffer_size=self.buffer_size)
        for s in ds:
            img = s['image'].convert('RGB')
            yield (self.transform(img) if self.transform else img), s['label']

train_dataset = HFStreamingImageNet('train',      transform=train_transform,
                                     shuffle=True, buffer_size=BUFFER_SIZE)
val_dataset   = HFStreamingImageNet('validation', transform=val_transform, shuffle=False)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True,
                               persistent_workers=True, prefetch_factor=2)
val_dataloader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=2, pin_memory=True,
                               persistent_workers=True, prefetch_factor=2)
print('Datasets ready.')


In [ ]:
# Cell 8 — Optional HPO with Optuna
# Each trial: HPO_EPOCHS epochs capped at HPO_SAMPLES_PER_EPOCH images.
# ~5-10 min per trial on A100.  Set N_HPO_TRIALS = 0 to skip.

def _hpo_objective(trial):
    lr  = trial.suggest_float('lr',             1e-2,  1.5e-1, log=True)
    wd  = trial.suggest_float('wd',             1e-4,  5e-3,   log=True)
    mom = trial.suggest_float('momentum',       0.90,  0.995)
    ls  = trial.suggest_float('label_smoothing',0.05,  0.15)

    mdl = resnet50(weights=None).to(device)
    opt = init_lr_weight_decay(
        mdl, lr, wd, bias_weight_decay=wd/2,
        momentum=mom, nestrov=True, bias_lr=True, elem_bias=True,
        warmup_epochs=warmup_epochs, prune_epoch=prune_epoch,
    )
    init_network(opt)
    sched = WarmupAutoJenks(
        opt, milestones=gsm_lr_boundaries, warmup_factor=0.5,
        warmup_iters=warmup_epochs, prune_epochs=prune_epoch,
        reset=False, rewind_epoch=None,
    )
    loss_hpo = nn.CrossEntropyLoss(label_smoothing=ls)
    acc_hpo  = Accuracy(task='multiclass', num_classes=1000).to(device)

    best_acc = 0.0
    for ep in range(1, HPO_EPOCHS + 1):
        mdl.train()
        seen = 0
        for X, y in train_dataloader:
            if seen >= HPO_SAMPLES_PER_EPOCH: break
            X, y = X.to(device), y.to(device)
            opt.zero_grad(); loss_hpo(mdl(X), y).backward(); opt.step()
            seen += X.size(0)
        sched.step()
        mdl.eval()
        ep_acc = val_n = 0
        with torch.inference_mode():
            for X, y in val_dataloader:
                if val_n * BATCH_SIZE >= 10_000: break
                X, y = X.to(device), y.to(device)
                ep_acc += acc_hpo(mdl(X), y).item(); val_n += 1
        proxy = ep_acc / max(val_n, 1)
        trial.report(proxy, ep)
        if trial.should_prune():
            raise __import__('optuna').exceptions.TrialPruned()
        best_acc = max(best_acc, proxy)

    del mdl, opt, sched; torch.cuda.empty_cache()
    return best_acc

if N_HPO_TRIALS > 0:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(
        direction='maximize',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=5),
    )
    study.optimize(_hpo_objective, n_trials=N_HPO_TRIALS, show_progress_bar=True)
    best = study.best_params
    print(f'Best HPO params: {best}')
    learning_rate, weight_decay = best['lr'], best['wd']
    momentum, label_smoothing   = best['momentum'], best['label_smoothing']
else:
    print('Skipping HPO — using default hyperparameters.')

print(f'lr={learning_rate:.4g}  wd={weight_decay:.4g}  '
      f'mom={momentum:.4g}  ls={label_smoothing:.3g}')


In [ ]:
# Cell 9 — Build model, optimizer, scheduler, EMA
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
experiment_name, model_name = 'ImageNet', 'ResNet50'

model    = resnet50(weights=None).to(device)
loss_fn  = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
accuracy = Accuracy(task='multiclass', num_classes=1000).to(device)
top5acc  = MulticlassAccuracy(num_classes=1000, top_k=5).to(device)

optimizer = init_lr_weight_decay(
    model, learning_rate, weight_decay,
    bias_weight_decay=bias_weight_decay,
    momentum=momentum, nestrov=nestrov, bias_lr=bias_lr, elem_bias=True,
    warmup_epochs=warmup_epochs, prune_epoch=prune_epoch,
)
init_network(optimizer)

scheduler = WarmupAutoJenks(
    optimizer, milestones=gsm_lr_boundaries,
    warmup_factor=0.5, warmup_iters=warmup_epochs,
    prune_epochs=prune_epoch, reset=False, rewind_epoch=None,
)

if USE_EMA:
    from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
    ema_model = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(EMA_DECAY))
else:
    ema_model = None

writer = SummaryWriter(f'runs/{timestamp}/{experiment_name}/{model_name}')

def _logfile(prefix):
    return os.path.join(OUTPUT_DIR, f'{prefix}_{timestamp}.txt')

train_filename    = _logfile('training_log')
val_filename      = _logfile('validation_log')
log_filename      = _logfile('log')
sparsity_filename = _logfile('sparsity_log')
prune_filename    = _logfile('prune_log')
debug_filename    = _logfile('debug_log')
jenks_filename    = _logfile('jenks_log')

print(f'Parameters : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print(f'Eff. batch : {BATCH_SIZE * accum_steps}')
print(f'EMA        : starts epoch {EMA_START_EPOCH} '
      f'({EPOCHS - EMA_START_EPOCH} averaging epochs)')


In [ ]:
# Cell 10 — Full training run
train_val_loop_HPO(
    model, train_dataloader, val_dataloader,
    optimizer, loss_fn, scheduler,
    accuracy, top5acc, writer, device,
    experiment_name, model_name, timestamp,
    train_filename=train_filename, val_filename=val_filename,
    log_filename=log_filename,     sparsity_filename=sparsity_filename,
    prune_filename=prune_filename, debug_filename=debug_filename,
    jenks_filename=jenks_filename,
    prune_count=0,     one_update=one_update,
    EPOCHS=EPOCHS,     sparsity=0.0,
    prune_epoch_list=[prune_epoch], prune_epoch=prune_epoch,
    prune_between=prune_between,    prune_ratio=prune_ratio,
    one_shot=one_shot, mask=mask,   mag_prune=True,
    bias_prune=bias_prune,          kill_velocity=kill_velocity,
    l2=False,          lambda_=0,   warmup_epochs=warmup_epochs,
    min_epochs=min_epochs,          elem_bias=True,
    accum_steps=accum_steps,
    ema_model=ema_model,
    ema_start_epoch=EMA_START_EPOCH,
)
writer.close()
print('Training complete.')


In [ ]:
# Cell 11 — Copy checkpoints and logs to Drive
import shutil, glob

for ckpt in glob.glob('models/*.pth'):
    dst = os.path.join(MODEL_DIR, os.path.basename(ckpt))
    shutil.copy2(ckpt, dst)
    print(f'Saved: {dst}')

if os.path.exists('runs'):
    runs_dst = '/content/drive/MyDrive/ResNet50_runs'
    shutil.copytree('runs', runs_dst, dirs_exist_ok=True)
    print(f'TensorBoard logs -> {runs_dst}')

print('Done.')
